# Barcelona Traffic Feature Engineering
**Period:** October 2017 – September 2018  
**Goal:** Compress millions of raw 5-minute traffic readings into a single memory-efficient feature table.  
Each street segment (`idTram`) gets one row with aggregated metrics per regulatory noise time block:
- **Day (Ld):** 07:00–20:59
- **Evening (Le):** 21:00–22:59
- **Night (Ln):** 23:00–06:59

**`estatActual` codes:** `0` = sensor offline, `1–2` = fluid traffic, `5` = congested.  
Zero-coded rows are retained for uptime accounting but excluded from percentage calculations.

In [1]:
import pandas as pd
import numpy as np
import glob
import os

## 1. File Discovery
Scan the traffic layer directory and select only the 12 monthly files within the target window (Oct 2017 – Sep 2018).  
Files for Oct–Dec 2018 live in the same `2018/` folder and are excluded by parsing the month number from the filename.

In [2]:
TRAFFIC_DIR = os.path.normpath(os.path.join('..', '..', 'layers', 'traffic'))

all_files = sorted(glob.glob(os.path.join(TRAFFIC_DIR, '**', '*_TRAMS_TRAMS.csv'), recursive=True))

def in_target_window(filepath):
    stem = os.path.basename(filepath)
    parts = stem.split('_')
    year, month = int(parts[0]), int(parts[1])
    return (year == 2017 and month >= 10) or (year == 2018 and month <= 9)

target_files = [f for f in all_files if in_target_window(f)]

print(f'Found {len(target_files)} files in target window:')
for f in target_files:
    print(f'  {os.path.basename(f)}')

Found 12 files in target window:
  2017_10_Octubre_TRAMS_TRAMS.csv
  2017_11_Novembre_TRAMS_TRAMS.csv
  2017_12_Desembre_TRAMS_TRAMS.csv
  2018_01_Gener_TRAMS_TRAMS.csv
  2018_02_Febrer_TRAMS_TRAMS.csv
  2018_03_Marc_TRAMS_TRAMS.csv
  2018_04_Abril_TRAMS_TRAMS.csv
  2018_05_Maig_TRAMS_TRAMS.csv
  2018_06_Juny_TRAMS_TRAMS.csv
  2018_07_Juliol_TRAMS_TRAMS.csv
  2018_08_Agost_TRAMS_TRAMS.csv
  2018_09_Setembre_TRAMS_TRAMS.csv


## 2. Processing Pipeline

The pipeline runs in three stages to keep peak RAM low:

1. **Per-file aggregation** – Read one monthly CSV at a time, assign time blocks, group by `idTram × time_block`, and compute intermediate counts. The raw frame is discarded immediately.
2. **Global consolidation** – Stack the 12 tiny monthly summaries and re-group to get 12-month totals.
3. **Pivot to wide format** – One row per `idTram`, one column group per time block.

### Why pooled standard deviation?
Standard deviations cannot be averaged across months. Instead, intermediate columns (`sum_estat`, `sum_sq_estat`) carry enough information to reconstruct the exact global sample std using the pooled variance formula:

`σ = sqrt( (Σx² − (Σx)² / n) / (n − 1) )`

In [3]:
def assign_time_block(hour_series):
    conditions = [
        (hour_series >= 7) & (hour_series <= 20),
        (hour_series >= 21) & (hour_series <= 22),
    ]
    return np.select(conditions, ['Day', 'Evening'], default='Night')

In [4]:
START = pd.Timestamp('2017-10-01')
END   = pd.Timestamp('2018-09-30 23:59:59')

def process_month(filepath):
    df = pd.read_csv(
        filepath,
        dtype={'idTram': 'int32', 'data': 'int64', 'estatActual': 'int8', 'estatPrevist': 'int8'}
    )

    df['ts'] = pd.to_datetime(df['data'].astype(str), format='%Y%m%d%H%M%S')
    df = df[(df['ts'] >= START) & (df['ts'] <= END)].copy()
    if df.empty:
        return pd.DataFrame()

    df['time_block'] = assign_time_block(df['ts'].dt.hour)
    df['estat_valid'] = df['estatActual'].where(df['estatActual'] != 0).astype('float32')

    agg = df.groupby(['idTram', 'time_block'], observed=True).agg(
        total_rows      = ('estatActual', 'size'),
        valid_rows      = ('estatActual', lambda x: (x != 0).sum()),
        fluid_count     = ('estatActual', lambda x: x.isin([1, 2]).sum()),
        congested_count = ('estatActual', lambda x: (x == 5).sum()),
        sum_estat       = ('estat_valid', 'sum'),
        sum_sq_estat    = ('estat_valid', lambda x: x.pow(2).sum()),
    ).reset_index()

    return agg

## 3. Batch Processing Loop
Each file is processed and immediately reduced to a small summary before the next file is loaded.

In [5]:
monthly_summaries = []

for fp in target_files:
    name = os.path.basename(fp)
    print(f'Processing {name} ...', end=' ', flush=True)
    summary = process_month(fp)
    if not summary.empty:
        monthly_summaries.append(summary)
        print(f'{len(summary):,} segment×block rows')
    else:
        print('EMPTY – skipped')

monthly_df = pd.concat(monthly_summaries, ignore_index=True)
print(f'\nStacked monthly summaries: {monthly_df.shape[0]:,} rows × {monthly_df.shape[1]} cols')

Processing 2017_10_Octubre_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2017_11_Novembre_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2017_12_Desembre_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_01_Gener_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_02_Febrer_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_03_Marc_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_04_Abril_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_05_Maig_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_06_Juny_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_07_Juliol_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_08_Agost_TRAMS_TRAMS.csv ... 1,602 segment×block rows
Processing 2018_09_Setembre_TRAMS_TRAMS.csv ... 1,599 segment×block rows

Stacked monthly summaries: 19,221 rows × 8 cols


## 4. Final Global Consolidation
Sum all intermediate columns across the 12 months, then derive the four final metrics:

| Column | Formula |
|---|---|
| `pct_fluid` | `fluid_count / valid_rows` |
| `pct_congested` | `congested_count / valid_rows` |
| `sensor_uptime_pct` | `valid_rows / total_rows` |
| `traffic_volatility` | pooled sample std of valid `estatActual` values |

In [6]:
global_agg = (
    monthly_df
    .groupby(['idTram', 'time_block'], as_index=False)
    [['total_rows', 'valid_rows', 'fluid_count', 'congested_count', 'sum_estat', 'sum_sq_estat']]
    .sum()
)

n = global_agg['valid_rows']

global_agg['pct_fluid']         = global_agg['fluid_count']     / n
global_agg['pct_congested']     = global_agg['congested_count'] / n
global_agg['sensor_uptime_pct'] = n / global_agg['total_rows']

numerator = global_agg['sum_sq_estat'] - global_agg['sum_estat'].pow(2) / n
global_agg['traffic_volatility'] = np.sqrt(numerator / (n - 1))

features = global_agg[[
    'idTram', 'time_block',
    'pct_fluid', 'pct_congested', 'traffic_volatility', 'sensor_uptime_pct'
]].copy()

print(f'Long-form feature table: {features.shape[0]:,} rows')
print(f'Unique idTram values:    {features["idTram"].nunique():,}')
print(f'Time blocks present:     {sorted(features["time_block"].unique())}')
features.head(9)

Long-form feature table: 1,602 rows
Unique idTram values:    534
Time blocks present:     ['Day', 'Evening', 'Night']


,idTram,time_block,pct_fluid,pct_congested,traffic_volatility,sensor_uptime_pct
0,1,Day,0.772630,0.005279,0.820653,0.938790
1,1,Evening,0.987780,0.000679,0.541149,0.924090
2,1,Night,0.999090,0.000000,0.286871,0.352856
3,2,Day,0.908792,0.000968,0.647269,0.947784
4,2,Evening,0.983729,0.002712,0.542601,0.925345
5,2,Night,0.994311,0.001264,0.254731,0.507702
6,3,Day,0.748743,0.001281,0.639599,0.930990
7,3,Evening,0.904282,0.000000,0.487392,0.498118
8,3,Night,0.993684,0.000000,0.320515,0.152439


## 5. Pivot to Wide Format
Transform from long format (one row per `idTram × time_block`) to wide format (one row per `idTram`).  
Column naming convention: `{block_prefix}_{metric}` — e.g. `day_pct_fluid`, `evening_traffic_volatility`, `night_sensor_uptime_pct`.

In [7]:
METRICS      = ['pct_fluid', 'pct_congested', 'traffic_volatility', 'sensor_uptime_pct']
BLOCK_PREFIX = {'Day': 'day', 'Evening': 'evening', 'Night': 'night'}

wide = features.pivot_table(
    index='idTram',
    columns='time_block',
    values=METRICS
)

wide.columns = [f'{BLOCK_PREFIX[block]}_{metric}' for metric, block in wide.columns]
wide = wide.reset_index()

ordered_cols = ['idTram']
for prefix in ['day', 'evening', 'night']:
    ordered_cols += [f'{prefix}_{m}' for m in METRICS]

wide = wide[ordered_cols]

print(f'Wide-format table: {wide.shape[0]:,} rows × {wide.shape[1]} columns')
print(f'Columns: {list(wide.columns)}')
wide.head()

Wide-format table: 534 rows × 13 columns
Columns: ['idTram', 'day_pct_fluid', 'day_pct_congested', 'day_traffic_volatility', 'day_sensor_uptime_pct', 'evening_pct_fluid', 'evening_pct_congested', 'evening_traffic_volatility', 'evening_sensor_uptime_pct', 'night_pct_fluid', 'night_pct_congested', 'night_traffic_volatility', 'night_sensor_uptime_pct']


,idTram,day_pct_fluid,day_pct_congested,day_traffic_volatility,day_sensor_uptime_pct,evening_pct_fluid,evening_pct_congested,evening_traffic_volatility,evening_sensor_uptime_pct,night_pct_fluid,night_pct_congested,night_traffic_volatility,night_sensor_uptime_pct
0,1,0.772630,0.005279,0.820653,0.938790,0.987780,0.000679,0.541149,0.924090,0.999090,0.000000,0.286871,0.352856
1,2,0.908792,0.000968,0.647269,0.947784,0.983729,0.002712,0.542601,0.925345,0.994311,0.001264,0.254731,0.507702
2,3,0.748743,0.001281,0.639599,0.930990,0.904282,0.000000,0.487392,0.498118,0.993684,0.000000,0.320515,0.152439
3,4,0.753924,0.008138,0.698794,0.947233,0.935015,0.000970,0.566782,0.646801,0.990814,0.001312,0.594143,0.244544
4,5,0.658947,0.256364,1.422203,0.958980,0.973394,0.012330,0.666745,0.966750,1.000000,0.000000,0.199152,0.318196


## 6. Validation
Confirm that percentages are in [0, 1] and that fluid + congested never exceed 100 %.

In [8]:
fluid_cols     = [c for c in wide.columns if c.endswith('_pct_fluid')]
congested_cols = [c for c in wide.columns if c.endswith('_pct_congested')]
uptime_cols    = [c for c in wide.columns if c.endswith('_sensor_uptime_pct')]

for f_col, c_col in zip(fluid_cols, congested_cols):
    combined = wide[f_col].fillna(0) + wide[c_col].fillna(0)
    assert (combined <= 1.0 + 1e-6).all(), f'pct_fluid + pct_congested > 1 in {f_col}'

for col in uptime_cols:
    assert wide[col].between(0, 1).all(), f'sensor_uptime_pct out of [0,1] in {col}'

print('All validation checks passed.')
print('\nSummary statistics:')
wide.describe().round(4)

All validation checks passed.

Summary statistics:


,idTram,day_pct_fluid,day_pct_congested,day_traffic_volatility,day_sensor_uptime_pct,evening_pct_fluid,evening_pct_congested,evening_traffic_volatility,evening_sensor_uptime_pct,night_pct_fluid,night_pct_congested,night_traffic_volatility,night_sensor_uptime_pct
count,534.0000,474.0000,474.0000,472.0000,534.0000,449.0000,449.0000,448.0000,534.0000,446.0000,446.0000,446.0000,534.0000
mean,279.5449,0.7986,0.0175,0.6089,0.6495,0.9251,0.0097,0.5250,0.5880,0.9413,0.0128,0.4784,0.2863
std,215.0710,0.2637,0.0424,0.3236,0.3949,0.1573,0.0397,0.3207,0.3923,0.1361,0.0594,0.3649,0.2914
min,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,134.2500,0.7557,0.0000,0.4177,0.1019,0.9366,0.0000,0.3820,0.0613,0.9513,0.0000,0.2482,0.0144
50%,267.5000,0.8997,0.0020,0.5892,0.8868,0.9791,0.0000,0.4969,0.7641,0.9888,0.0000,0.3981,0.1864
75%,401.7500,0.9714,0.0151,0.8011,0.9465,0.9962,0.0030,0.6278,0.9498,0.9985,0.0019,0.6324,0.4779
max,2004.0000,1.0000,0.3709,2.3979,0.9855,1.0000,0.4248,2.8284,0.9906,1.0000,0.6275,2.2812,0.9804


## 7. Export
Save the curated feature table. One row per `idTram`, 12 feature columns, ready to join with the noise model dataset.

In [10]:
OUT_PATH = os.path.normpath(os.path.join('..', '..', 'data', 'processed', 'barcelona_traffic_features_2017_2018.csv'))

wide.to_csv(OUT_PATH, index=False)

size_kb = os.path.getsize(OUT_PATH) / 1024
print(f'Saved:  {OUT_PATH}')
print(f'Size:   {size_kb:.1f} KB')
print(f'Shape:  {wide.shape[0]:,} rows × {wide.shape[1]} columns')

Saved:  ..\..\data\processed\barcelona_traffic_features_2017_2018.csv
Size:   95.0 KB
Shape:  534 rows × 13 columns
